# 6b · Event-centric open extraction — n-ary nodes + relations  (per article)

Reifies the article's facts into **n-ary nodes** — `Event`, `Observation`, `Claim` — and wires nb4's
already-typed, deduplicated entities to them as open `relations`. The entity set is **frozen**: only
ids from nb4's file may be arguments, which is precisely the constraint the single-call e2e baseline
(6c) does not operate under, and the reason the two are worth comparing.

An optional **second** call reviews the draft against the text and returns a repair patch — the same
pass 6c runs, so neither track gets a second look the other did not.

- **Config:** one line — `EXPERIMENT`; everything else comes from `configs/<EXPERIMENT>.json`.
- **Input:** `<id>/two_staged/1_entities/entities_typed<_isfx>[_consolidated].json` (nb4) + the raw
  article text, for the reference date and the evidence spans.
- **Output:** `<id>/two_staged/3_nary_graph/nary_graph<_osfx>.json` — the same shape as 6c's graph,
  so **nb7b** canonicalizes its relations and **nb8** renders it.

In [1]:
# --- resolve Phase2_LLM_Extraction root (works whether this notebook lives at the
# project root or under notebooks/<track>/) so results/ & dataset/ paths are stable.
from pathlib import Path
def _find_root(start: Path = None):
    start = (start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / 'results').is_dir() and (p / 'dataset').is_dir():
            return p
    return start
ROOT = _find_root()
# make Path writes create missing parent dirs (results/ now has sub-folders like
# two_staged/1_entities/); guarded so re-running the cell never double-wraps.
import pathlib as _pl
if not getattr(_pl.Path.write_text, '_mkparents', False):
    _wt, _wb = _pl.Path.write_text, _pl.Path.write_bytes
    def write_text(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wt(self, *a, **k)
    def write_bytes(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wb(self, *a, **k)
    write_text._mkparents = write_bytes._mkparents = True
    _pl.Path.write_text, _pl.Path.write_bytes = write_text, write_bytes

In [2]:
# ---- WHICH EXPERIMENT ---------------------------------------------------------------------
# ONE name decides the whole run. Every knob below comes from configs/<EXPERIMENT>.json — the SAME
# file every other notebook in both tracks reads, so a chain cannot drift apart stage to stage.
#   list what exists :  python -m experiment_config list          (run from notebooks/)
#   see one resolved :  python -m experiment_config show main
# $PHASE2_EXPERIMENT wins when it is set, so a headless batch of ablations (run_e2e.sh) can sweep
# experiments without editing a single notebook.
import os
EXPERIMENT = os.environ.get("PHASE2_EXPERIMENT") or "main"

import sys, pathlib, importlib
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "experiment_config.py").exists():
        sys.path.insert(0, str(_p)); break
import experiment_config as xcfg; importlib.reload(xcfg)

# Sets RESULTS_MODEL, BACKEND, GEMINI_MODEL, VARIANT, RUN_TAG, INPUT_TAG, SELECTED_ARTICLES and this
# section's own knobs: PROMPT, REVIEW, WORKERS, THINKING_BUDGET, MAX_OUTPUT_TOKENS.
CFG = xcfg.load(EXPERIMENT, globals(), "6b")
# A one-off try goes RIGHT HERE, below the load, where it wins over the file.

# --- derived paths -------------------------------------------------------------------------
from pathlib import Path

RESULTS_DIR = ROOT / "results" / RESULTS_MODEL
DATASET_DIR = ROOT / "dataset/articles"   # reads <id>.txt for the REFERENCE_DATE and evidence spans

# _isfx says which nb4 run's typed entities to reify over; _osfx tags what this notebook writes, and
# nb7b reads that tag back as its own INPUT_TAG. VARIANT names the reification prompt arm, exactly as
# it does for 6c — which is what lets the staged and e2e reified graphs be compared arm for arm.
_isfx = f"_{INPUT_TAG}" if INPUT_TAG else ""
_sfx  = f"_{VARIANT}" if VARIANT else ""
_osfx = _sfx + (f"_{RUN_TAG}" if RUN_TAG else "")
OUTPUT_NAME = f"two_staged/3_nary_graph/nary_graph{_osfx}.json"    # open, pre-canonicalization

# Which per-article entity file to use as the relation arguments. The consolidated file (nb4's global
# re-consolidation) is preferred when present, because that is the vocabulary the later stages match
# against; the raw incremental one is the fallback. Both carry the nb4 run's tag, so this stage can
# only ever read the entities of the run named by INPUT_TAG — mixing two nb4 runs' entities into one
# relation set is the failure this indirection exists to prevent.
ENTITY_FILES = [f"two_staged/1_entities/entities_typed{_isfx}_consolidated.json",
                f"two_staged/1_entities/entities_typed{_isfx}.json"]

def entity_path(aid):
    for name in ENTITY_FILES:
        p = RESULTS_DIR / aid / name
        if p.exists():
            return p
    return None

_available = sorted({p.relative_to(RESULTS_DIR).parts[0]
                     for p in RESULTS_DIR.glob(f"*/two_staged/1_entities/entities_typed{_isfx}*.json")},
                    key=xcfg.article_key)
if SELECTED_ARTICLES is None:
    ARTICLE_IDS = _available
else:
    missing = [a for a in SELECTED_ARTICLES if a not in _available]
    if missing:
        print("WARNING: skipping (no nb4 entity file for this INPUT_TAG):", missing)
    ARTICLE_IDS = [a for a in SELECTED_ARTICLES if a in _available]
assert ARTICLE_IDS, (
    f"No runnable articles. Looked for */two_staged/1_entities/entities_typed{_isfx}*.json — nb4's "
    f"output for INPUT_TAG={INPUT_TAG!r} under experiment {EXPERIMENT!r}. Run nb2 -> nb4 first.")
ARTICLE_ID = ARTICLE_IDS[0]                      # the single article the demo cell below uses

print(f"\nreads nb4 run {INPUT_TAG!r} | writes run {RUN_TAG!r} -> results/{RESULTS_MODEL}/<id>/{OUTPUT_NAME}")
print(f"{len(ARTICLE_IDS)} article(s): {ARTICLE_IDS}")

experiment 'art2-fix'  ·  section 6b (two_staged · event reification)
---------------------------------------------------------------------
  REPAIR of art2 in the gemini staged arm. nb6b truncated it: 62,912 thinking tokens against 2,610 of output, so the JSON never finished (finish_reason=MAX_TOKENS). Capped thinking for 6b, and nb7b grafts art2 onto the FINISHED 20-article vocabulary instead of rebuilding it — fresh=false so the vocabulary survives, inline_vocab_candidates=true so the model can reuse the terms already minted, and consolidate=false because a fresh pass would overwrite the cumulative merge report and strand the other 19 articles' references.
---------------------------------------------------------------------
  RESULTS_MODEL                'gemini-2.5-flash'
  BACKEND                      'gemini-2.5-flash'
  VARIANT                      'dedup'
  RUN_TAG                      'main_val_r1'
  INPUT_TAG                    'main_val_r1'
  REPEAT                       No

In [3]:
# --- Gemini client -----------------------------------------------------------------------
%pip install -q google-genai

import os, json, time, re
from google import genai
from google.genai import types as genai_types
from IPython.display import Markdown, display

GEMINI_API_KEY = "" or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Paste your key in GEMINI_API_KEY above, or set it in the env and restart the kernel."

client = genai.Client(api_key=GEMINI_API_KEY)
last_latency_s = None
LAST_USAGE = None

# Output token cap for one Gemini call.  None -> no limit (model maximum).  int -> cap the response.
# gemini-2.5-flash "thinking" tokens SHARE the output budget; uncapped thinking on a big article can
# starve the JSON answer and truncate it.  0 -> thinking OFF (all budget to the answer).
#
# These two DEFER TO THE CONFIG. Assigning them outright here silently reverted whatever
# xcfg.load() set one cell earlier, so `6b.max_output_tokens` and `6b.thinking_budget` never had any
# effect — which is how art2 died twice with byte-identical token counts (62,912 thinking, 2,610 of
# answer, truncated at the model's own 65,536 ceiling) despite a configured cap of 24,576.
MAX_OUTPUT_TOKENS = globals().get("MAX_OUTPUT_TOKENS", None)
THINKING_BUDGET = globals().get("THINKING_BUDGET", -1)

def _loads_lenient(text, resp=None):
    """Parse Gemini's JSON, tolerating code fences / trailing commas, with a CLEAR message on truncation."""
    t = (text or "").strip()
    if t.startswith("```"):
        t = re.sub(r"^```[a-zA-Z]*\n?", "", t)
        t = re.sub(r"\n?```$", "", t).strip()
    try:
        return json.loads(t)
    except json.JSONDecodeError:
        pass
    t2 = re.sub(r",(\s*[}\]])", r"\1", t)
    try:
        return json.loads(t2)
    except json.JSONDecodeError as e:
        fr = None
        try: fr = resp.candidates[0].finish_reason
        except Exception: pass
        raise ValueError(
            f"Gemini returned unparseable JSON (finish_reason={fr}; {len(t)} chars). "
            f"If finish_reason is MAX_TOKENS the output was TRUNCATED mid-JSON. Fixes: set THINKING_BUDGET=0 "
            f"(now {THINKING_BUDGET}) so thinking doesn't eat the budget, and/or raise MAX_OUTPUT_TOKENS "
            f"(now {MAX_OUTPUT_TOKENS}; None = model max). json error: {e}\n"
            f"...last 300 chars of output:\n{t[-300:]}"
        ) from None

_RETRY_STATUS = (429, 500, 502, 503, 504)
_RETRY_WORDS = ("UNAVAILABLE", "RESOURCE_EXHAUSTED", "INTERNAL", "high demand", "overloaded")


def _gemini_retryable(exc) -> bool:
    """A transient server-side failure, as opposed to a bad request worth not repeating."""
    code = getattr(exc, "code", None) or getattr(exc, "status_code", None)
    if code in _RETRY_STATUS:
        return True
    return any(w.lower() in str(exc).lower() for w in _RETRY_WORDS)


def _gemini_call_with_retry(fn, attempts=5, base=4):
    """Call `fn()`, retrying TRANSIENT API errors with backoff (4s, 8s, 16s, 32s).

    Without this one `503 UNAVAILABLE - This model is currently experiencing high demand` ends the
    whole stage: it killed nb4 two articles into twenty and took nb6b and nb7b down with it. Google's
    own message says such spikes are temporary, which is what a backoff is for. A 400 or 403 still
    fails at once — repeating a malformed or unauthorised request achieves nothing.
    """
    import time
    for attempt in range(attempts):
        try:
            return fn()
        except Exception as exc:                       # noqa: BLE001 - re-raised below
            if attempt == attempts - 1 or not _gemini_retryable(exc):
                raise
            wait = min(60, base * 2 ** attempt)
            code = getattr(exc, "code", None) or getattr(exc, "status_code", "?")
            print(f"    [retry {attempt + 1}/{attempts - 1}] {type(exc).__name__} {code} — "
                  f"waiting {wait}s", flush=True)
            time.sleep(wait)


def gemini_json(system_prompt, user_message, model=None, temperature=0, max_output_tokens="__default__"):
    """Call Gemini with JSON-mode output; return the parsed object, recording tokens + latency."""
    global last_latency_s, LAST_USAGE
    lim = MAX_OUTPUT_TOKENS if max_output_tokens == "__default__" else max_output_tokens
    cfg_kwargs = dict(system_instruction=system_prompt, temperature=temperature,
                      response_mime_type="application/json")
    if lim is not None:
        cfg_kwargs["max_output_tokens"] = lim
    if THINKING_BUDGET is not None and hasattr(genai_types, "ThinkingConfig"):
        cfg_kwargs["thinking_config"] = genai_types.ThinkingConfig(thinking_budget=THINKING_BUDGET)
    t0 = time.perf_counter()
    resp = _gemini_call_with_retry(lambda: client.models.generate_content(
        model=model or GEMINI_MODEL, contents=user_message,
        config=genai_types.GenerateContentConfig(**cfg_kwargs)))
    last_latency_s = time.perf_counter() - t0
    LAST_USAGE = nb_parallel.record_gemini_usage(resp, last_latency_s)   # prompt/output/THINKING tokens + latency
    return _loads_lenient(resp.text, resp)


# --- select the backend ---------------------------------------------------------------------
# Must stay LAST in this cell: the Gemini `gemini_json` above is redefined every time the cell
# runs, so whichever binding happens last wins. In a separate cell this would be silently undone.
# activate() sets gemini_json, GEMINI_MODEL, RESULTS_DIR, MAX_OUTPUT_TOKENS and THINKING_BUDGET.
import sys, pathlib, importlib
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "llm_backend.py").exists():
        sys.path.insert(0, str(_p)); break
import nb_parallel, llm_backend
importlib.reload(nb_parallel); importlib.reload(llm_backend)   # plain `import` is a no-op once cached
nb_parallel.install()           # thread-aware stdout + the token/latency ledger used by usage_report

llm_backend.activate(BACKEND, ROOT, globals(), results=RESULTS_MODEL)


Note: you may need to restart the kernel to use updated packages.
backend: GEMINI gemini-2.5-flash  ->  results/gemini-2.5-flash/


{'backend': 'gemini-2.5-flash',
 'kind': 'gemini',
 'model': 'gemini-2.5-flash',
 'results_dir': PosixPath('/Users/marwah/Documents/Thesis/TriplesExtractionFromWebLLMs/Phase2_LLM_Extraction/results/gemini-2.5-flash')}

## 1. Load the article's entities (the participants relations can point at)

Identical to nb6, so the n-ary graph and the binary graph draw from the **same** entity node set. Each
entity is an id-stamped card (`E1, E2, …`). The n-ary nodes minted below (`EV.., OBS.., CL..`) are the
*additional* nodes that relations may also point at.

In [4]:
def _demojibake(s):
    """Repair UTF-8-decoded-as-latin-1 mojibake (e.g. 'Â£3.73m' -> '£3.73m'). Gated on tell-tale markers."""
    if isinstance(s, str) and any(m in s for m in ("Ã", "Â", "â€")):
        try:
            return s.encode("latin-1").decode("utf-8")
        except (UnicodeDecodeError, UnicodeEncodeError):
            return s
    return s

def load_entities(aid):
    """Read the article's final entity nodes (nb4) and build id-stamped participant cards."""
    p = entity_path(aid)
    assert p is not None, f"no entity file for {aid} (expected one of {ENTITY_FILES} under results/{aid}/)"
    data = json.loads(p.read_text())
    cards = []
    for i, e in enumerate(data.get("entities", []), 1):
        cards.append({"id": f"E{i}", "text": _demojibake(e.get("canonical_text")),
                      "type": e.get("chosen_type"), "aliases": [_demojibake(a) for a in e.get("aliases", [])],
                      "type_source": e.get("type_source"), "type_parent": e.get("type_parent"),
                      "type_vocab_id": e.get("vocab_id")})
    return p.name, cards

def _reference_date(text):
    """Parse the 'Date: DD/MM/YYYY' dateline -> ISO-8601 (YYYY-MM-DD), or None."""
    m = re.search(r"^Date:\s*(\d{2})/(\d{2})/(\d{4})", text, re.MULTILINE)
    if not m:
        return None
    d, mth, y = m.groups()
    return f"{y}-{mth}-{d}"

_fname, _cards = load_entities(ARTICLE_ID)
print(f"{ARTICLE_ID}: {len(_cards)} entities from {_fname}")
for c in _cards[:8]:
    print(f"  {c['id']:<4} {c['text']!r:<28} :: {c['type']}")

art2: 135 entities from entities_typed_main_val_r1_consolidated.json
  E1   'Taylor Swift'               :: Person
  E2   'Travis Kelce'               :: Person
  E3   'Lena Dunham'                :: Person
  E4   'Jack Antonoff'              :: Person
  E5   'Ryan Weng'                  :: Person
  E6   'Rusty Carbaug'              :: Person
  E7   'Zohran Mamdani'             :: Person
  E8   'Rachel Antonoff'            :: Person


## 2. Open extraction — n-ary nodes + relations

One Gemini call per article. The model (1) mints **n-ary nodes** for the events / observations / claims
the article is about, and (2) extracts **open relations** — free-form `[subject, relation_phrase,
object]` triples where subject and object are entity or n-ary-node ids. No fixed role set and no
canonicalization here: the `relation_phrase`s and n-ary `type`s are matched to schema.org / a separate
n-ary vocabulary and minted downstream (nb7), exactly like the binary path.

In [5]:
PROMPT_NARY = """You are an expert event-centric open information extraction system.
You are given a TEXT, its REFERENCE_DATE (publication date, ISO-8601), and a list of ENTITIES already extracted from it (each with an id, surface text, type, aliases).

GOAL: Build ONE concise, comprehensive, well-connected event-centric graph in two parts:
(1) N-ARY NODES — new reified nodes for the events, observations, and claims the TEXT is about; and
(2) RELATIONS — open, free-form directed relations tying nodes together, each with the relation phrase that best fits the TEXT. You are NOT given a fixed set of relations: invent the concise phrase that fits. After extraction, define each unique relation phrase.

N-ARY NODE KINDS (create nodes of these three kinds only):
- "Event": something that happens or is scheduled/forecast to happen.
- "Observation": a stated measurement, statistic, quantity, or quantified change.
- "Claim": a statement explicitly attributed to a source (someone says/states/reports/claims something).
Each n-ary node: {"id" (unique, e.g. EV1 / OBS1 / CL1), "node_kind", "type" (a concise free-form class label — PascalCase — as specific as the TEXT warrants), "label" (a SHORT natural title for the event/observation/claim — a few words, no packed-in facts), "is_main" (bool), "evidence" (verbatim span(s)), "confidence" (0-1)}.

RELATIONS: each is a directed triple {"subject_id","relation_phrase","object_id","evidence","confidence"} where subject_id and object_id are ids of ENTITIES or N-ARY NODES. Orient it so it reads subject --relation_phrase--> object. Choose the concise relation_phrase that best captures the link.

RULES:
1. Identify the article's MAIN event(s) and set "is_main": true on them.
2. Tie every participant of a stated fact to its n-ary node with a relation, and also capture the stable entity-to-entity relations. Aim for a single connected graph — do not leave a relevant node dangling.
3. Keep every fact's context connected to it: attach all the details that qualify a fact — its value, time, place, cause, qualifier, and source — to THAT fact's n-ary node, so the fact and its context form one connected cluster reachable in a few hops. Never leave a relation that adds context to a fact disconnected from that fact's node.
4. Keep each n-ary node's label a SHORT, natural title that reads as the event/observation/claim (a few words); do NOT pack facts into it - values, places, times, causes, quantities, sources and participants are RELATIONS, not label text. Any entity you name in the label MUST also be attached as a relation (the label never replaces an edge).
5. Give every Observation and Claim at least one relation to the Event it describes, and relate each Event to the other Events it is linked to in the TEXT if applicable (e.g. "part of" / "caused by" / "led to" / "followed"), so nothing hangs free of the event backbone.
6. When a breakdown gives several values that each belong to a distinct item (a category, group, period, or round), relate EACH value to its item AND link each item to the shared parent fact/event, so the whole breakdown stays in ONE connected cluster. Do NOT attach all the values and all the items as separate spokes on one summarizing node — that leaves you unable to tell which value goes with which item.
7. subject_id and object_id MUST be existing ENTITY or N-ARY NODE ids; never relate a node to itself.
8. Only omit entities that are not part of the article's actual content (navigation, "related"/"read more" links, boilerplate).
9. Extract only what the TEXT explicitly states. Use REFERENCE_DATE to resolve relative dates where possible.

OUTPUT strict JSON:
{
  "nary_nodes": [ {"id","node_kind","type","label","is_main","evidence":[...],"confidence"} ],
  "relations": [ {"subject_id","relation_phrase","object_id","evidence","confidence"} ],
  "relation_definitions": [ {"relation_phrase","definition"} ]
}
"""

# The reification prompts, one per arm. Only one exists today — the staged track has never needed a
# second — but the arm is NAMED and selected by the config's `prompt` key, exactly as in 6c, so a new
# arm is a dict entry rather than a cell you have to remember to re-run last.
NARY_PROMPTS = {"nary": PROMPT_NARY}
assert PROMPT in NARY_PROMPTS, f"config prompt={PROMPT!r}; choose one of {list(NARY_PROMPTS)}"
NARY_SYSTEM_PROMPT = NARY_PROMPTS[PROMPT]        # extract_nary reads this global at call time
print(f"reification prompt: {PROMPT!r} ({len(NARY_SYSTEM_PROMPT):,} chars)  |  review pass: {REVIEW}")


def _fmt_entities(cards):
    return "\n".join(
        f'{c["id"]}\t{c["text"]}\t({c["type"]})' + (f'\taliases: {c["aliases"]}' if c["aliases"] else "")
        for c in cards)

def extract_nary(aid, write=True):
    """Open extraction of n-ary nodes + relations for one article -> results/<id>/nary_graph.json."""
    fname, cards = load_entities(aid)
    text = _demojibake((DATASET_DIR / f"{aid}.txt").read_text())
    ref_date = _reference_date(text)

    user_message = (
        f"REFERENCE_DATE: {ref_date}\n\n"
        f"ENTITIES (id\ttext\t(type)):\n{_fmt_entities(cards)}\n\n"
        f"TEXT:\n{text}"
    )
    t_article = time.perf_counter()
    out = gemini_json(NARY_SYSTEM_PROMPT, user_message)
    # per-THREAD usage, not the LAST_USAGE global: with a parallel batch that global holds whichever
    # call finished last, which need not be this article's.
    ext_usage  = nb_parallel.last_usage()             # capture NOW; the review call replaces it
    nary_nodes = out.get("nary_nodes", [])
    relations  = out.get("relations", [])

    # resolve each relation's endpoints to text/type (entity card OR n-ary node) so nb7 & the viz can
    # read subject/object surface + type, exactly like nb6's relation records.
    node_text = {c["id"]: (c["text"], c["type"]) for c in cards}
    for n in nary_nodes:
        node_text[n["id"]] = (n.get("label"), n.get("type"))
    for r in relations:
        r["subject_text"], r["subject_type"] = node_text.get(r.get("subject_id"), (None, None))
        r["object_text"], r["object_type"]   = node_text.get(r.get("object_id"), (None, None))

    graph = {
        "article_id":           aid,
        "reference_date":       ref_date,
        "entity_file":          fname,
        "entities":             cards,
        "nary_nodes":           nary_nodes,
        "relations":            relations,           # OPEN relation_phrases — canonicalized in nb7
        "relation_definitions": out.get("relation_definitions", []),
    }

    review_usage = None
    if REVIEW:                                        # optional 2nd LLM pass: check/repair vs TEXT
        graph = review_graph(aid, text, graph)
        review_usage = nb_parallel.last_usage()

    # deterministic normalization: drop duplicate + self-loop edges (the review pass can reintroduce
    # an edge that already exists). Entities are NOT touched — they are nb4's, frozen by design.
    graph, dd = _dedup_relations(graph)
    nary_nodes, relations = graph["nary_nodes"], graph["relations"]
    if dd["relations_dropped"] or dd["selfloops_dropped"]:
        print(f"    dedup {aid}: dropped {dd['relations_dropped']} duplicate + "
              f"{dd['selfloops_dropped']} self-loop relation(s)")

    def _tot(u, k): return (u or {}).get(k) or 0
    graph["token_usage"] = {
        "extraction": ext_usage, "review": review_usage,
        "combined": {k: _tot(ext_usage, k) + _tot(review_usage, k)
                     for k in ("prompt_tokens", "output_tokens", "thoughts_tokens", "total_tokens")}}

    if write:
        p = RESULTS_DIR / aid / OUTPUT_NAME
        p.write_text(json.dumps(graph, indent=2, ensure_ascii=False))
        print(f"  wrote {p}  ({len(nary_nodes)} n-ary nodes, {len(relations)} relations)  "
              f"[{time.perf_counter() - t_article:.1f}s]")
    return graph

reification prompt: 'nary' (3,895 chars)  |  review pass: True


In [6]:
# ── OPTIONAL review/repair pass (SEPARATE 2nd LLM call) ──────────────────────────────────────────
# The staged twin of 6c's review pass, and the reason both tracks can be compared on equal terms: if
# only the e2e arm got a second look at the text, the comparison would be measuring the review, not
# the decomposition. Same contract — the pass returns a small PATCH rather than a rewritten graph, so
# it can only touch what it names and cannot silently drop the rest.
#
# ONE deliberate difference from 6c: this pass may NOT add entities. The staged pipeline's whole
# premise is that nb4's typed, deduplicated entity set is FROZEN before relations are extracted, so a
# review that minted new entities would quietly dissolve the very property being tested. A relation
# whose endpoint is not in that set is dropped by the guard instead.
#
# REVIEW comes from the config (section "6b"); the `no-review` experiment turns it off in both tracks
# at once, which is what makes that ablation meaningful.

REVIEW_PROMPT = """You are an expert knowledge-graph reviewer and repairer. You are given a news TEXT, the fixed list of ENTITIES extracted from it, and a DRAFT graph of n-ary nodes and relations built over those entities. Do NOT rebuild the graph. Inspect it against the TEXT and return a small PATCH that fixes ONLY these problems:

1. MISSING FACTS — add any relation the TEXT states or unambiguously entails that the draft omitted (entity-to-entity or entity-to-n-ary-node). Every subject_id/object_id MUST be an id that already exists: an ENTITY id from the given list, or an n-ary node id from the draft or from add_nary_nodes. You may NOT invent entities — if a fact needs a participant that is not in the ENTITIES list, leave that fact out. Never invent a relation the TEXT does not support.
2. OVER-REIFICATION — if an n-ary node encodes what is really a stable, direct fact between two entities, collapse it: list the node id in remove_nary_nodes and re-express the fact as a direct entity-to-entity relation in add_relations (carry its evidence). Only when no information is lost.
3. REDUNDANT / WRONG edges — remove exact-duplicate edges, and fix a wrong direction (remove the wrong one and add the corrected one).
4. EVIDENCE GAPS — every relation's and n-ary node's evidence spans must EXPLICITLY name its subject and object (for a node, every entity named in its label); a pronoun or definite description ("this", "the country", "some areas") does NOT count. Where the naming sentence is missing, return corrected evidence — the original spans plus the nearest VERBATIM complete sentence(s) that name the entity — via fix_relation_evidence / fix_nary_node_evidence.

Keep every existing correct fact; do not remove anything unless it is redundant or you are re-expressing it. Return ONLY valid JSON — a PATCH, using empty lists where there is nothing to change:
{
  "add_nary_nodes":    [ {"id","node_kind","type","label","is_main":false,"evidence":[],"confidence":1.0} ],
  "remove_nary_nodes": [ "<id>" ],
  "add_relations":     [ {"subject_id","relation_phrase","object_id","evidence":[],"confidence":1.0} ],
  "remove_relations":  [ {"subject_id","relation_phrase","object_id"} ],
  "fix_relation_evidence":  [ {"subject_id","relation_phrase","object_id","evidence":[]} ],
  "fix_nary_node_evidence": [ {"id","evidence":[]} ],
  "add_relation_definitions": [ {"relation_phrase","definition","synonyms":[]} ],
  "notes": "one short line on what you changed"
}"""


def _resolve_endpoint_texts(graph):
    """Recompute subject/object text+type on every relation from the entity cards + n-ary nodes."""
    ref = {e["id"]: (e.get("text"), e.get("type")) for e in graph["entities"]}
    ref.update({n["id"]: (n.get("label"), n.get("type")) for n in graph["nary_nodes"]})
    for r in graph["relations"]:
        r["subject_text"], r["subject_type"] = ref.get(r.get("subject_id"), (None, None))
        r["object_text"],  r["object_type"]  = ref.get(r.get("object_id"),  (None, None))
    return graph


def _guard_endpoints(graph):
    """Drop any relation whose subject or object id is not a known entity or n-ary node. This is what
    enforces the frozen entity set: a review that reaches for an entity nb4 never produced simply
    loses that edge, rather than growing the entity set behind the pipeline's back."""
    ids = {e["id"] for e in graph["entities"]} | {n["id"] for n in graph["nary_nodes"]}
    kept, dropped = [], []
    for r in graph["relations"]:
        (kept if r.get("subject_id") in ids and r.get("object_id") in ids else dropped).append(r)
    graph["relations"] = kept
    return graph, dropped


def _union_evidence(old, new):
    """Append new spans not already present — augment evidence, never shrink it, so an EVIDENCE-GAP
    fix can only ADD the missing naming sentence even if the model forgets to echo the originals."""
    merged = list(old or [])
    for s in (new or []):
        if s not in merged:
            merged.append(s)
    return merged


def _dedup_relations(graph):
    """Drop exact-duplicate edges (evidence unioned) and any self-loop. Entities are left alone: they
    are nb4's output and their ids are the contract this stage is built on."""
    seen, newrels, selfloops = {}, [], 0
    for r in graph.get("relations", []):
        if r.get("subject_id") == r.get("object_id"):
            selfloops += 1; continue
        k = (r.get("subject_id"), r.get("relation_phrase"), r.get("object_id"))
        if k in seen:
            seen[k]["evidence"] = _union_evidence(seen[k].get("evidence"), r.get("evidence"))
        else:
            seen[k] = r; newrels.append(r)
    dropped = len(graph.get("relations", [])) - len(newrels) - selfloops
    graph["relations"] = newrels
    _resolve_endpoint_texts(graph)
    return graph, {"relations_dropped": dropped, "selfloops_dropped": selfloops}


def _apply_patch(graph, patch):
    nodes = {n["id"]: n for n in graph["nary_nodes"]}
    for n in (patch.get("add_nary_nodes") or []):
        nodes.setdefault(n.get("id"), n)                    # never overwrite an existing node
    rm = set(patch.get("remove_nary_nodes") or [])
    for nid in rm:
        nodes.pop(nid, None)
    rmrel = {(x.get("subject_id"), x.get("relation_phrase"), x.get("object_id"))
             for x in (patch.get("remove_relations") or [])}
    def _key(r): return (r.get("subject_id"), r.get("relation_phrase"), r.get("object_id"))
    rels = [r for r in graph["relations"]                    # drop edges on removed nodes + removals
            if r.get("subject_id") not in rm and r.get("object_id") not in rm and _key(r) not in rmrel]
    rels += (patch.get("add_relations") or [])
    rel_by_key = {}
    for r in rels:
        rel_by_key.setdefault(_key(r), r)
    for fx in (patch.get("fix_relation_evidence") or []):
        r = rel_by_key.get((fx.get("subject_id"), fx.get("relation_phrase"), fx.get("object_id")))
        if r is not None:
            r["evidence"] = _union_evidence(r.get("evidence"), fx.get("evidence"))
    for fx in (patch.get("fix_nary_node_evidence") or []):
        n = nodes.get(fx.get("id"))
        if n is not None:
            n["evidence"] = _union_evidence(n.get("evidence"), fx.get("evidence"))
    rdefs = graph.get("relation_definitions", []); seen_r = {d.get("relation_phrase") for d in rdefs}
    rdefs += [d for d in (patch.get("add_relation_definitions") or [])
              if d.get("relation_phrase") not in seen_r]
    graph["nary_nodes"] = list(nodes.values())
    graph["relations"] = rels
    graph["relation_definitions"] = rdefs
    return graph


def review_graph(aid, text, graph):
    """SECOND LLM call: TEXT + ENTITIES + draft graph -> PATCH -> apply -> guard -> resolve texts."""
    before  = (len(graph["nary_nodes"]), len(graph["relations"]))
    payload = {k: graph[k] for k in ("nary_nodes", "relations")}
    patch = gemini_json(REVIEW_PROMPT,
                        f"TEXT:\n{text}\n\nENTITIES (id\ttext\t(type)):\n"
                        + _fmt_entities(graph["entities"])
                        + "\n\nDRAFT GRAPH (JSON):\n" + json.dumps(payload, ensure_ascii=False))
    if not isinstance(patch, dict):
        # A failed review is a REVIEWED article with an empty patch, not an unreviewed one.
        print(f"    [warn] {aid}: review returned no patch object; keeping draft")
        graph["review_stats"] = {"patch_parsed": False,
                                 "nary_nodes_before": before[0], "nary_nodes_after": before[0],
                                 "relations_before": before[1], "relations_after": before[1]}
        return _resolve_endpoint_texts(graph)
    graph = _apply_patch(graph, patch)
    graph, dropped = _guard_endpoints(graph)
    graph = _resolve_endpoint_texts(graph)
    graph["review_notes"] = patch.get("notes")
    after = (len(graph["nary_nodes"]), len(graph["relations"]))
    n_fix = len(patch.get("fix_relation_evidence") or []) + len(patch.get("fix_nary_node_evidence") or [])
    # SAVED, not just printed — same accounting as 6c's, minus the entity counts (this track's
    # entities come from nb2/nb4 and the review pass cannot add them).
    graph["review_stats"] = {
        "nary_nodes_before": before[0], "nary_nodes_after": after[0],
        "relations_before": before[1], "relations_after": after[1],
        "added_nary_nodes": len(patch.get("add_nary_nodes") or []),
        "added_relations": len(patch.get("add_relations") or []),
        "removed_nary_nodes": len(patch.get("remove_nary_nodes") or []),
        "removed_relations": len(patch.get("remove_relations") or []),
        "fixed_evidence": n_fix,
        "guard_dropped": len(dropped),
        "patch_parsed": True,
    }
    print(f"    review {aid}: nodes {before[0]}->{after[0]}  rels {before[1]}->{after[1]}  "
          f"(+{len(patch.get('add_relations') or [])} rel, "
          f"collapsed {len(patch.get('remove_nary_nodes') or [])} node, fixed {n_fix} ev, "
          f"guard-dropped {len(dropped)})")
    return graph

## 3. Inspect the demo article

In [7]:
# DEMO — one article, printed rather than written. Off by default so a batch run does not extract
# ARTICLE_ID a second time; set DEMO = True to inspect a single article interactively.
DEMO = False

if DEMO:
    g = extract_nary(ARTICLE_ID)
    by_id = {c["id"]: c["text"] for c in g["entities"]}
    by_id.update({n["id"]: n["label"] for n in g["nary_nodes"]})

    print("-" * 80)
    print(f'{len(g["nary_nodes"])} n-ary nodes:')
    for n in g["nary_nodes"]:
        star = " *MAIN*" if n.get("is_main") else ""
        print(f'  {n["id"]:<5} {n["node_kind"]:<12} <{n.get("type")}>{star}  "{n["label"]}"')
    print("-" * 80)
    print(f'{len(g["relations"])} relations (open relation_phrase — canonicalized later in nb7):')
    for r in g["relations"]:
        s = by_id.get(r["subject_id"], r["subject_id"]); o = by_id.get(r["object_id"], r["object_id"])
        print(f'  {s!r}  --{r["relation_phrase"]}-->  {o!r}   (conf {r.get("confidence")})')
    print("-" * 80)
    print("relation_phrase definitions:")
    for d in g["relation_definitions"]:
        print(f'  {d.get("relation_phrase"):<26} {d.get("definition","")}')

## 4. Batch — process the selected articles (`ARTICLE_IDS`)

In [8]:
# Open extraction for each article. Re-runs overwrite. Canonicalization of the relation_phrases into
# the n-ary property vocabulary happens next, in nb7b.
#
# Articles are independent — nothing is carried from one to the next — so WORKERS of them run at
# once, exactly as in 6c. Each article costs 1 LLM call, or 2 with REVIEW on. Each article's prints
# are buffered and shown as one block when it finishes; a failure is reported and skipped rather
# than killing the batch.
nb_parallel.reset_usage()          # the report below covers only the calls THIS run makes
_res  = nb_parallel.pmap(extract_nary, ARTICLE_IDS, workers=WORKERS)
_runs = [(r.item, r.value) for r in _res if r.ok]
print(f"\ndone -- extracted {len(_runs)}/{len(ARTICLE_IDS)} article(s) -> results/<id>/{OUTPUT_NAME}")

# --- per-article token usage (extraction + review), persisted per experiment -------------------
def _u(g, which, k): return ((g.get("token_usage", {}) or {}).get(which) or {}).get(k) or 0
_grand = {k: sum(_u(g, "combined", k) for _, g in _runs)
          for k in ("prompt_tokens", "output_tokens", "thoughts_tokens", "total_tokens")}
_tok_path = RESULTS_DIR / f"_global/token_usage_nb6b_nary{_osfx}.json"
_tok_path.write_text(json.dumps(
    {"method": "staged_reification", "experiment": EXPERIMENT, "prompt": PROMPT, "review": REVIEW,
     "model": GEMINI_MODEL, "thinking_budget": THINKING_BUDGET,
     "per_article": [{"article": aid, **(g.get("token_usage", {}) or {})} for aid, g in _runs],
     "totals": _grand}, indent=2, ensure_ascii=False))
print(f"wrote {_tok_path}")

# --- what this run cost: calls, prompt / output / THINKING tokens, and elapsed time ------------
# wall clock is the batch's real elapsed time; with WORKERS>1 the summed LLM time is larger, and the
# ratio between them is the overlap the thread pool actually bought.
nb_parallel.usage_report(f"nb6b · event reification · {EXPERIMENT}",
                         wall_seconds=nb_parallel.LAST_WALL_SECONDS,
                         experiment=EXPERIMENT, run_tag=RUN_TAG)

1 item(s), running sequentially

[1/1] art2
   FAILED: ValueError: Gemini returned unparseable JSON (finish_reason=FinishReason.MAX_TOKENS; 50275 chars). If finish_reason is MAX_TOKENS the output was TRUNCATED mid-JSON. Fixes: set THINKING_BUDGET=0 (now 8192) so thinking doesn't eat the budget, and/or raise MAX_OUTPUT_TOKENS (now 24576; None = model max). json error: Expecting value: line 1989 column 20 (char 50275)
...last 300 chars of output:

      "subject_id": "OBS13",
      "relation_phrase": "occurs in",
      "object_id": "E120",
      "evidence": [
        "through Manhattan"
      ],
      "confidence": 1.0
    },
    {
      "subject_id": "OBS13",
      "relation_phrase": "involves",
      "object_id": "E34",
      "evidence": [

done — 0/1 ok in 103.4s; FAILED: art2

done -- extracted 0/1 article(s) -> results/<id>/two_staged/3_nary_graph/nary_graph_dedup_main_val_r1.json
wrote /Users/marwah/Documents/Thesis/TriplesExtractionFromWebLLMs/Phase2_LLM_Extraction/results/gemini-

{'label': 'nb6b · event reification · art2-fix',
 'totals': {'calls': 1,
  'prompt_tokens': 4173,
  'output_tokens': 16371,
  'thoughts_tokens': 8189,
  'total_tokens': 28733,
  'seconds': 103.4},
 'wall_seconds': 103.40826104200096,
 'per_tag': {'art2': {'calls': 1,
   'seconds': 103.4,
   'prompt_tokens': 4173,
   'output_tokens': 16371,
   'thoughts_tokens': 8189,
   'total_tokens': 28733}},
 'calls': [{'prompt_tokens': 4173,
   'output_tokens': 16371,
   'thoughts_tokens': 8189,
   'total_tokens': 28733,
   'seconds': 103.40233887499198,
   'tag': 'art2'}],
 'experiment': 'art2-fix',
 'run_tag': 'main_val_r1'}

## 5. Run metrics — every number this stage produced, in one file

`run_metrics.extraction` re-reads the graphs just written and reports graph size per article, node
kinds, main events, distinct relation phrases, evidence density, **fragmentation** (largest connected
component share + isolated nodes) and cost — the **same metrics, in the same file format, as 6c**,
which is what makes the staged and e2e reified graphs directly comparable.

In [9]:
# Collect + persist this stage's numbers. Pure reading — no LLM call, safe to re-run at any time.
import importlib, run_metrics
importlib.reload(run_metrics)

_m = run_metrics.extraction(RESULTS_DIR, ARTICLE_IDS, sfx=_osfx, track="two_staged",
                            experiment=EXPERIMENT)
run_metrics.show(_m)

# save() also CLOSES this stage in the run manifest — the record of WHICH run these numbers
# came from: the resolved config, the git commit, the article order, every output file and
# the token spend.  results/<model>/_runs/<run_id>.json  ·  python -m run_manifest list
print("\nsaved ->", run_metrics.save(_m, RESULTS_DIR, _osfx, "two_staged",
                                     section="6b", articles=ARTICLE_IDS))

EXTRACTION  ·  two_staged  ·  experiment 'art2-fix'
  articles                           0
  main_events                        0
  distinct_entity_types              0
  distinct_relation_phrases          0
  mean_largest_component_share       None
  isolated_nodes                     0
  mean_evidence_spans                None
  reviewed_articles                  0
  review_ran                         False
  NOT overwriting metrics_extraction_dedup_main_val_r1.json: it covers 19 article(s), this run only 0.
  Recompute over the full list to refresh it (run_metrics.extraction(...) with every article, then save).

saved -> /Users/marwah/Documents/Thesis/TriplesExtractionFromWebLLMs/Phase2_LLM_Extraction/results/gemini-2.5-flash/_global/metrics_extraction_dedup_main_val_r1.json
